# 06 — LoRA/PEFT Fine-Tuning of BGE-M3 for Portuguese Legal Retrieval

This notebook performs the **domain adaptation** of BGE-M3 using Portuguese legal
retrieval examples and preserves the outputs from the completed training run.

The important point is that this is not only a training notebook. It documents a
complete experiment:

```text
baseline → LoRA attachment → triplet training → validation → checkpoint selection
```

## Observed result of this run

The best adapter was obtained at **epoch 2**. On semantic validation queries,
`MRR@10` increased from **0.8472** for the original BGE-M3 to **0.9791** with
LoRA.

These are validation results, not final test results. They are used to select the
checkpoint; the untouched test set must still be used for the final comparison.

## Pipeline position

```text
01 Extraction → 02 Parsing → 03 Dataset → 04 Baseline
                                      ↓
                           05 Hard negatives
                                      ↓
                           06 LoRA fine-tuning  ← here
                                      ↓
                              best adapter
                                      ↓
                           07 Final evaluation
```

## Learning objective

For every semantic query, the model sees a relevant passage and a difficult
non-relevant passage mined by the baseline model:

```text
Query
 ├── Positive      → should move closer
 └── Hard negative → should move farther away
```

LoRA adapts only a small set of parameters while the pretrained BGE-M3 weights
remain frozen.

## 1. Install dependencies

The experiment combines Sentence Transformers with PEFT.

- `sentence-transformers` provides the embedding pipeline and `TripletLoss`;
- `transformers` provides the underlying XLM-R/BGE-M3 architecture;
- `accelerate` supports the training stack and device execution;
- `peft` implements LoRA;
- `torch` provides tensor operations and CUDA support.

The outputs in this notebook were generated with the environment shown in the
CUDA/package check below. Reproducing a machine-learning experiment should include
both the model configuration and the software environment.

In [ ]:
# Run this once if necessary, then restart the Jupyter kernel.
#
# %pip install -U sentence-transformers transformers "accelerate>=1.1.0" peft pandas numpy tqdm

## 2. Imports

The imports combine model training, PEFT, retrieval evaluation and experiment
tracking.

The output below contains two warnings worth distinguishing from errors.

`TqdmWarning: IProgress not found` affects how progress bars are rendered inside
Jupyter; it does not invalidate training.

The Sentence Transformers `DeprecationWarning` indicates that an import path will
change in a future release. The code still executed successfully in this run.

For teaching purposes, this is a useful distinction:

```text
Warning → execution can continue, but something may need attention
Error   → execution stops or the requested operation fails
```

In [2]:
from pathlib import Path
from collections import defaultdict, Counter
import gc
import json
import math
import random
import time

import numpy as np
import pandas as pd
import torch

from torch.utils.data import DataLoader
from sentence_transformers import SentenceTransformer, InputExample, losses

from peft import LoraConfig, TaskType, get_peft_model

import accelerate
import transformers
import sentence_transformers
import peft

C:\Users\user\AppData\Roaming\Python\Python314\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\user\AppData\Local\Temp\ipykernel_14492\4062886102.py:14: DeprecationWarning: Importing from 'sentence_transformers.losses' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.losses' instead.
  from sentence_transformers import SentenceTransformer, InputExample, losses


## 3. CUDA and package check

This cell confirms that PyTorch is actually CUDA-enabled.

The observed environment was:

```text
PyTorch              2.13.0+cu130
CUDA build           13.0
CUDA available       True
GPU                   RTX 4050 Laptop GPU
Total VRAM           6.00 GB
Free VRAM initially  4.95 GB
```

This check is more important than `nvidia-smi` alone. The NVIDIA driver can detect
the GPU while Python is still using a CPU-only PyTorch package.

Here CUDA is correctly available, so the experiment can proceed on the GPU.

The initial 4.95 GB free VRAM is measured before loading BGE-M3.

In [3]:
print("PyTorch:", torch.__version__)
print("CUDA build:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
print("Accelerate:", accelerate.__version__)
print("Transformers:", transformers.__version__)
print("Sentence Transformers:", sentence_transformers.__version__)
print("PEFT:", peft.__version__)

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA is not available. This notebook is intended for GPU training."
    )

print("GPU:", torch.cuda.get_device_name(0))

free, total = torch.cuda.mem_get_info()

print(f"Free VRAM : {free / 1024**3:.2f} GB")
print(f"Total VRAM: {total / 1024**3:.2f} GB")

PyTorch: 2.13.0+cu130
CUDA build: 13.0
CUDA available: True
Accelerate: 1.14.0
Transformers: 5.15.0
Sentence Transformers: 5.7.0
PEFT: 0.20.0
GPU: NVIDIA GeForce RTX 4050 Laptop GPU
Free VRAM : 4.95 GB
Total VRAM: 6.00 GB


## 4. Project paths

The output confirms that the notebook is operating inside the
`legal-embeddings-finetuning-pt` project and separates:

```text
data/dataset_v2       → retrieval dataset
data/hard_negatives   → mined training negatives
models/...             → LoRA checkpoints
results/...            → experiment metrics and metadata
```

This separation is useful because model artefacts, experimental results and source
datasets have different roles and lifecycles.

In [4]:
PROJECT_ROOT = Path("..")

DATASET_DIR = PROJECT_ROOT / "data" / "dataset_v2"
HARD_NEGATIVE_DIR = PROJECT_ROOT / "data" / "hard_negatives"

MODEL_DIR = PROJECT_ROOT / "models" / "bge-m3-legal-pt-lora"
CHECKPOINT_DIR = MODEL_DIR / "checkpoints"
BEST_MODEL_DIR = MODEL_DIR / "best"

RESULTS_DIR = PROJECT_ROOT / "results" / "finetuning_lora"

MODEL_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

PASSAGES_PATH = DATASET_DIR / "passages.jsonl"
VALIDATION_QUERIES_PATH = DATASET_DIR / "validation_queries.jsonl"
VALIDATION_QRELS_PATH = DATASET_DIR / "validation_qrels.jsonl"

MINED_NEGATIVES_PATH = (
    HARD_NEGATIVE_DIR / "train_hard_negatives.jsonl"
)

print("Dataset       :", DATASET_DIR.resolve())
print("Hard negatives:", HARD_NEGATIVE_DIR.resolve())
print("Model output  :", MODEL_DIR.resolve())
print("Results       :", RESULTS_DIR.resolve())

Dataset       : C:\Users\user\Documents\GitHub\legal-embeddings-finetuning-pt\data\dataset_v2
Hard negatives: C:\Users\user\Documents\GitHub\legal-embeddings-finetuning-pt\data\hard_negatives
Model output  : C:\Users\user\Documents\GitHub\legal-embeddings-finetuning-pt\models\bge-m3-legal-pt-lora
Results       : C:\Users\user\Documents\GitHub\legal-embeddings-finetuning-pt\results\finetuning_lora


## 5. Experiment configuration

The completed run used:

```text
Base model           BAAI/bge-m3
Device               CUDA
Maximum sequence     256 tokens
Training batch       1
Epochs               2
LoRA rank            8
LoRA alpha           16
Target modules       query, key, value
```

The batch size of 1 is deliberately conservative for a 6 GB GPU.

`rank = 8` defines the capacity of each low-rank adapter. `alpha = 16` scales its
contribution. LoRA is attached to the query, key and value projections of the
self-attention mechanism.

These values form part of the experiment definition and should be reported with
the final results.

In [5]:
MODEL_NAME = "BAAI/bge-m3"

RANDOM_SEED = 42

MAX_SEQ_LENGTH = 256

TRAIN_BATCH_SIZE = 1
ENCODE_BATCH_SIZE = 4

NUM_EPOCHS = 2

LEARNING_RATE = 2e-4
WARMUP_RATIO = 0.10
WEIGHT_DECAY = 0.01

NEGATIVES_PER_QUERY = 1

TRIPLET_MARGIN = 0.20
TOP_K = 10

MODEL_SELECTION_METRIC = "mrr_at_10"

USE_AMP = True

# LoRA configuration
LORA_R = 8
LORA_ALPHA = 16
LORA_DROPOUT = 0.05

# XLM-R / BGE-M3 attention projection modules.
LORA_TARGET_MODULES = [
    "query",
    "key",
    "value",
]

DEVICE = "cuda"

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
torch.cuda.manual_seed_all(RANDOM_SEED)

print("Model               :", MODEL_NAME)
print("Device              :", DEVICE)
print("Maximum sequence    :", MAX_SEQ_LENGTH)
print("Training batch size :", TRAIN_BATCH_SIZE)
print("Epochs              :", NUM_EPOCHS)
print("LoRA rank           :", LORA_R)
print("LoRA alpha          :", LORA_ALPHA)
print("LoRA targets        :", LORA_TARGET_MODULES)

# Reproducibility note:
# Seeds control the main software-level sources of randomness. Exact GPU-level
# reproducibility can still depend on CUDA kernels, hardware and library versions.


Model               : BAAI/bge-m3
Device              : cuda
Maximum sequence    : 256
Training batch size : 1
Epochs              : 2
LoRA rank           : 8
LoRA alpha          : 16
LoRA targets        : ['query', 'key', 'value']


## 6. JSONL loader

All dataset records are read from JSON Lines files.

Each line is parsed independently, making JSONL convenient for large machine-
learning datasets and easy to inspect manually.

Explicit parsing errors are preferable to silently skipping malformed training
examples.

In [6]:
def read_jsonl(path):
    records = []

    with path.open("r", encoding="utf-8") as f:
        for line_number, line in enumerate(f, start=1):
            line = line.strip()

            if not line:
                continue

            try:
                records.append(json.loads(line))
            except json.JSONDecodeError as exc:
                raise ValueError(
                    f"Invalid JSON in {path.name}, line {line_number}: {exc}"
                )

    return records

## 7. Load the experimental data

The observed dataset contains:

```text
Passages             2,088
Mined train queries  4,224
Validation queries     516
Validation qrels       632
```

The training query pool contains:

```text
Semantic   3,165
Reference  1,059
```

A qrel is a ground-truth relevance judgement linking a query to a passage.

There are more validation qrels than validation queries because one query can have
multiple relevant passages, for example when a legal article was divided into
several chunks.

In [7]:
passages = read_jsonl(PASSAGES_PATH)
mined_records = read_jsonl(MINED_NEGATIVES_PATH)

validation_queries = read_jsonl(VALIDATION_QUERIES_PATH)
validation_qrels = read_jsonl(VALIDATION_QRELS_PATH)

passage_by_id = {
    passage["passage_id"]: passage
    for passage in passages
}

print("Passages           :", len(passages))
print("Mined train queries:", len(mined_records))
print("Validation queries :", len(validation_queries))
print("Validation qrels   :", len(validation_qrels))

print(
    "Query types:",
    Counter(record["query_type"] for record in mined_records)
)

Passages           : 2088
Mined train queries: 4224
Validation queries : 516
Validation qrels   : 632
Query types: Counter({'semantic': 3165, 'reference': 1059})


## 8. Keep only semantic training queries

The output shows:

```text
Semantic training queries  3,165
Reference queries excluded 1,059
```

This is intentional.

A reference query such as `O que estabelece o artigo 6.º do CIVA?` contains
explicit lexical identifiers. A semantic query requires the embedding model to
match the legal concept itself.

The fine-tuning experiment therefore focuses on the more demanding semantic
retrieval task.

In [8]:
semantic_records = [
    record
    for record in mined_records
    if record["query_type"] == "semantic"
]

print("Semantic training queries :", len(semantic_records))
print(
    "Reference queries excluded:",
    len(mined_records) - len(semantic_records)
)

Semantic training queries : 3165
Reference queries excluded: 1059


## 9. Build hard-negative training triplets

The notebook creates **3,165 training triplets** — one for every selected semantic
training query.

Each triplet contains:

```text
anchor   = semantic query
positive = relevant legal passage
negative = hard negative mined in notebook 05
```

The hard negative is especially useful because the original BGE-M3 already found
it deceptively similar to the query.

Training therefore concentrates on distinctions that the baseline model actually
found difficult.

In [9]:
training_records = []

for record in semantic_records:
    relevant_ids = sorted(record["relevant_passage_ids"])
    negatives = record["hard_negatives"][:NEGATIVES_PER_QUERY]

    if not relevant_ids or not negatives:
        continue

    for negative_index, negative in enumerate(negatives):
        positive_id = relevant_ids[
            negative_index % len(relevant_ids)
        ]

        positive = passage_by_id[positive_id]

        training_records.append({
            "query_id": record["query_id"],
            "query": record["query"],
            "document_id": record["document_id"],
            "source_article_id": record["source_article_id"],
            "positive_passage_id": positive_id,
            "positive": positive["text"],
            "negative_passage_id": negative["passage_id"],
            "negative_source_article_id": negative["source_article_id"],
            "negative": negative["text"],
            "negative_similarity": negative["similarity"],
            "negative_baseline_rank": negative["baseline_rank"],
            "negative_same_document": negative["same_document"],
        })

print("Training triplets:", len(training_records))

# The resulting 3,165 triplets correspond to the semantic training queries
# reported above. Each triplet contains the supervision used by TripletLoss.


Training triplets: 3165


## 10. Validate the training data

The result:

```text
Training-data errors: 0
```

is an important quality-control result.

It confirms that the triplets satisfy the implemented integrity rules before
training begins — in particular, positives and negatives do not conflict under
the article-level relevance assumptions.

Contrastive learning is sensitive to false negatives, so this validation is part
of the experimental methodology rather than merely defensive programming.

In [10]:
errors = []

for record in training_records:
    if record["positive_passage_id"] == record["negative_passage_id"]:
        errors.append("positive_equals_negative")

    if record["source_article_id"] == record["negative_source_article_id"]:
        errors.append("same_article_negative")

    if not record["query"].strip():
        errors.append("empty_query")

    if not record["positive"].strip():
        errors.append("empty_positive")

    if not record["negative"].strip():
        errors.append("empty_negative")

print("Training-data errors:", len(errors))
assert not errors

Training-data errors: 0


## 11. Create training examples and batches

The run contains:

```text
Training examples  3,165
Batches per epoch  3,165
```

Because `TRAIN_BATCH_SIZE = 1`, each batch contains one triplet.

Therefore, one epoch consists of 3,165 optimisation steps/batches through the
training set.

A small batch reduces VRAM pressure, although it can make training slower than on
a larger GPU.

In [11]:
train_examples = [
    InputExample(
        texts=[
            record["query"],
            record["positive"],
            record["negative"],
        ]
    )
    for record in training_records
]

train_dataloader = DataLoader(
    train_examples,
    shuffle=True,
    batch_size=TRAIN_BATCH_SIZE,
)

print("Training examples :", len(train_examples))
print("Batches per epoch :", len(train_dataloader))

Training examples : 3165
Batches per epoch : 3165


## 12. Prepare semantic validation data

From the validation set, **387 semantic queries** are selected.

These queries are never used for gradient updates.

Their role is model selection:

```text
TRAIN
→ changes LoRA weights

VALIDATION
→ measures retrieval quality
→ selects the best epoch
```

This distinction prevents us from choosing a checkpoint simply because it achieved
a low loss on examples it had already seen.

In [12]:
semantic_validation_queries = [
    query
    for query in validation_queries
    if query["query_type"] == "semantic"
]

relevant_validation = defaultdict(set)

for qrel in validation_qrels:
    if qrel.get("relevance", 0) > 0:
        relevant_validation[
            qrel["query_id"]
        ].add(qrel["passage_id"])

print(
    "Semantic validation queries:",
    len(semantic_validation_queries)
)

Semantic validation queries: 387


## 13. Retrieval metrics used during validation

The experiment reuses the same retrieval metrics as the baseline.

`Recall@K` measures how much relevant content appears within the first K results.

`MRR@10` focuses on the rank of the first relevant result:

```text
relevant at rank 1 → 1.00
relevant at rank 2 → 0.50
relevant at rank 5 → 0.20
```

`nDCG@10` evaluates the quality of the ranking while discounting relevant results
that appear lower in the list.

The checkpoint-selection criterion is **MRR@10**. Training loss is not used to
decide which epoch is best.

In [13]:
def recall_at_k(ranked_ids, relevant_ids, k):
    if not relevant_ids:
        return 0.0

    return (
        len(set(ranked_ids[:k]) & relevant_ids)
        / len(relevant_ids)
    )


def reciprocal_rank_at_k(ranked_ids, relevant_ids, k):
    for rank, passage_id in enumerate(ranked_ids[:k], start=1):
        if passage_id in relevant_ids:
            return 1.0 / rank

    return 0.0


def ndcg_at_k(ranked_ids, relevant_ids, k):
    dcg = 0.0

    for rank, passage_id in enumerate(ranked_ids[:k], start=1):
        if passage_id in relevant_ids:
            dcg += 1.0 / math.log2(rank + 1)

    ideal_hits = min(len(relevant_ids), k)

    if ideal_hits == 0:
        return 0.0

    idcg = sum(
        1.0 / math.log2(rank + 1)
        for rank in range(1, ideal_hits + 1)
    )

    return dcg / idcg

## 14. Semantic validation retrieval

Validation encodes the complete passage corpus and the semantic validation queries
using the current state of the model.

Because embeddings are normalised, cosine similarity can be calculated efficiently
with matrix multiplication:

```text
query embeddings × passage embeddingsᵀ
```

The resulting similarity scores produce a ranking of all passages for every query.

This process is repeated after each epoch because LoRA changes the embedding space.
Using passage embeddings calculated before training would therefore evaluate the
wrong model state.

In [14]:
def evaluate_semantic_validation(
    model,
    passages,
    queries,
    relevance_map,
    batch_size=ENCODE_BATCH_SIZE,
    top_k=TOP_K,
):
    passage_embeddings = model.encode(
        [passage["text"] for passage in passages],
        batch_size=batch_size,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True,
    )

    query_embeddings = model.encode(
        [query["query"] for query in queries],
        batch_size=batch_size,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True,
    )

    passage_embeddings = np.asarray(
        passage_embeddings,
        dtype=np.float32
    )

    query_embeddings = np.asarray(
        query_embeddings,
        dtype=np.float32
    )

    scores = query_embeddings @ passage_embeddings.T

    values = {
        "recall_at_1": [],
        "recall_at_5": [],
        "recall_at_10": [],
        "mrr_at_10": [],
        "ndcg_at_10": [],
    }

    for query_index, query in enumerate(queries):
        row = scores[query_index]
        candidate_count = min(top_k, len(passages))

        top_indices = np.argpartition(
            -row,
            candidate_count - 1
        )[:candidate_count]

        top_indices = top_indices[
            np.argsort(-row[top_indices])
        ]

        ranked_ids = [
            passages[index]["passage_id"]
            for index in top_indices
        ]

        relevant_ids = relevance_map[query["query_id"]]

        values["recall_at_1"].append(
            recall_at_k(ranked_ids, relevant_ids, 1)
        )
        values["recall_at_5"].append(
            recall_at_k(ranked_ids, relevant_ids, 5)
        )
        values["recall_at_10"].append(
            recall_at_k(ranked_ids, relevant_ids, 10)
        )
        values["mrr_at_10"].append(
            reciprocal_rank_at_k(ranked_ids, relevant_ids, 10)
        )
        values["ndcg_at_10"].append(
            ndcg_at_k(ranked_ids, relevant_ids, 10)
        )

    del passage_embeddings, query_embeddings, scores

    gc.collect()
    torch.cuda.empty_cache()

    return {
        metric: float(np.mean(metric_values))
        for metric, metric_values in values.items()
    }

## 15. Load the pretrained BGE-M3

The original BGE-M3 loads successfully with an embedding dimension of **1,024**.

After loading the base model:

```text
Free VRAM ≈ 2.82 GB
```

Compared with the initial 4.95 GB, this shows that a substantial part of the
6 GB GPU is already occupied before training begins.

The Hugging Face message about unauthenticated Hub requests concerns download rate
limits; it does not affect the validity of the loaded model.

The `FutureWarning` indicates a renamed API method and likewise does not invalidate
the result.

In [15]:
gc.collect()
torch.cuda.empty_cache()

model = SentenceTransformer(
    MODEL_NAME,
    device=DEVICE
)

model.max_seq_length = MAX_SEQ_LENGTH

print(
    "Embedding dimension:",
    model.get_sentence_embedding_dimension()
)

free, total = torch.cuda.mem_get_info()

print(
    f"Free VRAM after base model load: "
    f"{free / 1024**3:.2f} GB"
)

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 19385.48it/s]


Embedding dimension: 1024
Free VRAM after base model load: 2.82 GB


C:\Users\user\AppData\Local\Temp\ipykernel_14492\862951143.py:13: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  model.get_sentence_embedding_dimension()


## 16. Measure the local baseline

Before attaching LoRA, the original BGE-M3 is evaluated with exactly the same
validation function that will be used after training.

Observed semantic validation baseline:

| Metric | BGE-M3 baseline |
|---|---:|
| Recall@1 | 0.7063 |
| Recall@5 | 0.9426 |
| Recall@10 | 0.9580 |
| MRR@10 | 0.8472 |
| nDCG@10 | 0.8738 |

This local baseline is the correct reference for the following epoch comparisons
because the passages, semantic queries, sequence length and retrieval implementation
are identical.

At this point no LoRA parameters exist yet.

In [16]:
baseline_validation_metrics = evaluate_semantic_validation(
    model,
    passages,
    semantic_validation_queries,
    relevant_validation,
)

print("\nSemantic validation baseline")

for metric, value in baseline_validation_metrics.items():
    print(f"{metric:14}: {value:.4f}")

Batches: 100%|██████████| 97/97 [00:02<00:00, 38.96it/s]



Semantic validation baseline
recall_at_1   : 0.7063
recall_at_5   : 0.9426
recall_at_10  : 0.9580
mrr_at_10     : 0.8472
ndcg_at_10    : 0.8738


## 17. Inspect the underlying transformer

The output identifies the underlying architecture as:

```text
XLMRobertaModel
```

and confirms the presence of modules such as:

```text
attention.self.query
attention.self.key
attention.self.value
```

This inspection explains why the LoRA configuration can safely target the module
names `query`, `key` and `value`.

It is better to inspect the real architecture than to assume that module names are
identical across transformer families.

In [17]:
transformer_module = model[0]

hf_model = transformer_module.auto_model

print("Underlying model type:")
print(type(hf_model))

print("\nExample attention-related module names:")

shown = 0

for name, module in hf_model.named_modules():
    if any(
        token in name
        for token in [".query", ".key", ".value"]
    ):
        print(name)

        shown += 1

        if shown >= 20:
            break

Underlying model type:
<class 'transformers.models.xlm_roberta.modeling_xlm_roberta.XLMRobertaModel'>

Example attention-related module names:
encoder.layer.0.attention.self.query
encoder.layer.0.attention.self.key
encoder.layer.0.attention.self.value
encoder.layer.1.attention.self.query
encoder.layer.1.attention.self.key
encoder.layer.1.attention.self.value
encoder.layer.2.attention.self.query
encoder.layer.2.attention.self.key
encoder.layer.2.attention.self.value
encoder.layer.3.attention.self.query
encoder.layer.3.attention.self.key
encoder.layer.3.attention.self.value
encoder.layer.4.attention.self.query
encoder.layer.4.attention.self.key
encoder.layer.4.attention.self.value
encoder.layer.5.attention.self.query
encoder.layer.5.attention.self.key
encoder.layer.5.attention.self.value
encoder.layer.6.attention.self.query
encoder.layer.6.attention.self.key


## 18. Attach LoRA adapters

The output confirms:

```text
LoRA adapters attached.
```

LoRA augments selected linear layers with low-rank trainable updates while keeping
the original pretrained matrices frozen.

Conceptually:

```text
BGE-M3 pretrained weight W
          +
small trainable LoRA update ΔW
          ↓
adapted transformation
```

This is parameter-efficient fine-tuning: the complete base model still participates
in inference and backpropagation, but only the adapter parameters are optimised.

In [18]:
lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    target_modules=LORA_TARGET_MODULES,
    task_type=TaskType.FEATURE_EXTRACTION,
)

peft_model = get_peft_model(
    hf_model,
    lora_config
)

# Replace the Hugging Face model inside the
# Sentence Transformers transformer module.
transformer_module.auto_model = peft_model

print("LoRA adapters attached.")

# PEFT freezes the base parameters and injects trainable low-rank matrices
# into the selected attention projections.


LoRA adapters attached.


## 19. Verify trainable parameters

The output confirms the intended PEFT configuration:

```text
Trainable parameters  1,179,648
Total parameters      568,934,400
Trainable percentage  0.2073%
```

Only about **0.21%** of the model parameters are trainable.

Equivalently, roughly **99.79% of the parameters remain frozen**.

This is the central practical advantage of LoRA in this experiment. Instead of
optimising all ~569 million parameters, the GPU only needs gradients and optimiser
state for approximately 1.18 million trainable parameters.

In [19]:
trainable_params = 0
total_params = 0

for parameter in model.parameters():
    total_params += parameter.numel()

    if parameter.requires_grad:
        trainable_params += parameter.numel()

trainable_ratio = (
    100 * trainable_params / total_params
)

print(f"Trainable parameters : {trainable_params:,}")
print(f"Total parameters     : {total_params:,}")
print(f"Trainable percentage : {trainable_ratio:.4f}%")

peft_model.print_trainable_parameters()

if trainable_ratio > 5:
    print(
        "\nWARNING: More than 5% of parameters are trainable. "
        "Check the LoRA configuration before training."
    )

# This check verifies that LoRA is configured as intended before training:
# only ~0.21% of the complete model should receive gradient updates.


Trainable parameters : 1,179,648
Total parameters     : 568,934,400
Trainable percentage : 0.2073%
trainable params: 1,179,648 || all params: 568,934,400 || trainable%: 0.2073


## 20. GPU memory after LoRA attachment

The observed CUDA state is:

```text
Free VRAM   2.77 GB
Allocated   2.13 GB
Reserved    2.16 GB
Total       6.00 GB
```

Notice that attaching LoRA does not suddenly free most of the GPU.

The full BGE-M3 still has to remain in memory for the forward pass. The saving comes
mainly from avoiding gradients and optimiser states for the frozen base parameters.

This is why LoRA makes training feasible even though the loaded model itself still
occupies substantial VRAM.

In [20]:
gc.collect()
torch.cuda.empty_cache()

free, total = torch.cuda.mem_get_info()

print(f"Free VRAM : {free / 1024**3:.2f} GB")
print(f"Total VRAM: {total / 1024**3:.2f} GB")
print(
    f"Allocated : "
    f"{torch.cuda.memory_allocated() / 1024**3:.2f} GB"
)
print(
    f"Reserved  : "
    f"{torch.cuda.memory_reserved() / 1024**3:.2f} GB"
)

Free VRAM : 2.77 GB
Total VRAM: 6.00 GB
Allocated : 2.13 GB
Reserved  : 2.16 GB


## 21. Configure TripletLoss

The run contains **3,165 steps per epoch** and **316 warmup steps**, corresponding
to approximately 10% of one epoch.

TripletLoss compares three embeddings:

```text
query → anchor
correct passage → positive
confusing wrong passage → negative
```

With cosine distance and a margin of 0.20, the model is penalised when the positive
is not sufficiently closer to the query than the negative.

As training progresses, correctly separated triplets contribute less loss.

A low training loss therefore means that the training objective is being satisfied.
It does not, by itself, establish that retrieval improved on unseen legal articles.

In [21]:
train_loss = losses.TripletLoss(
    model=model,
    distance_metric=losses.TripletDistanceMetric.COSINE,
    triplet_margin=TRIPLET_MARGIN,
)

warmup_steps_per_epoch = max(
    1,
    int(len(train_dataloader) * WARMUP_RATIO)
)

print("Steps per epoch:", len(train_dataloader))
print("Warmup steps   :", warmup_steps_per_epoch)

# Warmup is expressed in optimiser steps. Here 316 steps are approximately
# 10% of the 3,165 batches processed in one epoch.


Steps per epoch: 3165
Warmup steps   : 316


## 22. Train one epoch at a time and validate

The training loop deliberately calls the trainer for one epoch at a time.

This allows the following sequence:

```text
Epoch 1
  ↓
semantic validation
  ↓
checkpoint decision
  ↓
Epoch 2
  ↓
semantic validation
  ↓
checkpoint decision
```

The outputs below are the actual completed run.

### Epoch 1

Training took approximately **1,709.3 seconds**, or **28 min 29 s**.

Validation produced:

| Metric | Baseline | Epoch 1 |
|---|---:|---:|
| Recall@1 | 0.7063 | **0.8782** |
| Recall@5 | 0.9426 | **1.0000** |
| Recall@10 | 0.9580 | **1.0000** |
| MRR@10 | 0.8472 | **0.9733** |
| nDCG@10 | 0.8738 | **0.9802** |

Epoch 1 therefore became the first best LoRA checkpoint.

### Epoch 2

Training took approximately **1,696.1 seconds**, or **28 min 16 s**.

Validation improved again:

| Metric | Epoch 1 | Epoch 2 |
|---|---:|---:|
| Recall@1 | 0.8782 | **0.8885** |
| Recall@5 | 1.0000 | **1.0000** |
| Recall@10 | 1.0000 | **1.0000** |
| MRR@10 | 0.9733 | **0.9791** |
| nDCG@10 | 0.9802 | **0.9845** |

Epoch 2 therefore replaced epoch 1 as the best checkpoint.

### Important interpretation

The gain from epoch 1 to epoch 2 is smaller than the initial gain from baseline to
epoch 1. This is normal: the first adaptation pass produced the largest change and
the second refined it further.

The correct selection is still epoch 2 because validation `MRR@10` increased.

In [22]:
history = []

best_metric = baseline_validation_metrics[
    MODEL_SELECTION_METRIC
]

best_epoch = 0

print(
    f"Initial {MODEL_SELECTION_METRIC}: "
    f"{best_metric:.4f}"
)

Initial mrr_at_10: 0.8472


In [23]:
for epoch in range(1, NUM_EPOCHS + 1):

    print("\n" + "=" * 90)
    print(f"EPOCH {epoch}/{NUM_EPOCHS}")

    gc.collect()
    torch.cuda.empty_cache()

    free, total = torch.cuda.mem_get_info()

    print(
        f"Free VRAM at epoch start: "
        f"{free / 1024**3:.2f} GB"
    )

    start = time.time()

    model.fit(
        train_objectives=[
            (train_dataloader, train_loss)
        ],
        epochs=1,
        warmup_steps=warmup_steps_per_epoch,
        optimizer_params={
            "lr": LEARNING_RATE
        },
        weight_decay=WEIGHT_DECAY,
        use_amp=USE_AMP,
        show_progress_bar=True,
        save_best_model=False,
    )

    training_seconds = time.time() - start

    print(
        f"Epoch training time: "
        f"{training_seconds:.1f} s"
    )

    gc.collect()
    torch.cuda.empty_cache()

    print(
        "\nRunning semantic validation retrieval..."
    )

    validation_start = time.time()

    metrics = evaluate_semantic_validation(
        model,
        passages,
        semantic_validation_queries,
        relevant_validation,
    )

    validation_seconds = (
        time.time() - validation_start
    )

    history.append({
        "epoch": epoch,
        "training_seconds": training_seconds,
        "validation_seconds": validation_seconds,
        **metrics,
    })

    print()

    for metric, value in metrics.items():
        print(f"{metric:14}: {value:.4f}")

    current_metric = metrics[
        MODEL_SELECTION_METRIC
    ]

    epoch_adapter_dir = (
        CHECKPOINT_DIR
        / f"epoch_{epoch:02d}_adapter"
    )

    # Save only the PEFT adapter.
    peft_model.save_pretrained(
        epoch_adapter_dir
    )

    if current_metric > best_metric:
        best_metric = current_metric
        best_epoch = epoch

        print(
            f"\nNew best LoRA checkpoint: "
            f"epoch {epoch}"
        )

        peft_model.save_pretrained(
            BEST_MODEL_DIR
        )

    else:
        print(
            "\nNo improvement over "
            "the current best checkpoint."
        )

# The validation metrics printed after each epoch are the checkpoint-selection
# signal. The test set is deliberately absent from this training loop.



EPOCH 1/2
Free VRAM at epoch start: 2.77 GB


Step,Training Loss
500,0.057992
1000,0.022856
1500,0.032772
2000,0.014807
2500,0.012651
3000,0.010631


Epoch training time: 1709.3 s

Running semantic validation retrieval...


Batches: 100%|██████████| 97/97 [00:05<00:00, 16.20it/s]



recall_at_1   : 0.8782
recall_at_5   : 1.0000
recall_at_10  : 1.0000
mrr_at_10     : 0.9733
ndcg_at_10    : 0.9802

New best LoRA checkpoint: epoch 1

EPOCH 2/2
Free VRAM at epoch start: 2.74 GB


Step,Training Loss
500,0.008560
1000,0.023208
1500,0.010377
2000,0.010988
2500,0.013178
3000,0.009894


Epoch training time: 1696.1 s

Running semantic validation retrieval...


Batches: 100%|██████████| 97/97 [00:07<00:00, 13.56it/s]



recall_at_1   : 0.8885
recall_at_5   : 1.0000
recall_at_10  : 1.0000
mrr_at_10     : 0.9791
ndcg_at_10    : 0.9845

New best LoRA checkpoint: epoch 2


## 23. Training history

The table below stores the exact per-epoch results.

A useful observation is that both epochs required roughly the same training time:

```text
Epoch 1 ≈ 1709 s
Epoch 2 ≈ 1696 s
```

Validation took only tens of seconds because it performs inference rather than
backpropagation.

The retrieval metrics improve from epoch 1 to epoch 2, so there is no evidence in
these two epochs that additional training has yet harmed validation performance.
That does not imply that indefinitely adding epochs would continue to help.

In [24]:
history_df = pd.DataFrame(history)
display(history_df)

,epoch,training_seconds,validation_seconds,recall_at_1,recall_at_5,recall_at_10,mrr_at_10,ndcg_at_10
0,1,1709.328803,37.909627,0.878165,1.0,1.0,0.973299,0.980189
1,2,1696.087760,42.938475,0.888501,1.0,1.0,0.979113,0.984501


## 24. Quantify improvement over the baseline

The comparison table below calculates:

```text
delta = LoRA validation metric - baseline validation metric
```

For the selected epoch 2:

| Metric | Baseline | LoRA epoch 2 | Absolute gain |
|---|---:|---:|---:|
| Recall@1 | 0.7063 | 0.8885 | **+0.1822** |
| Recall@5 | 0.9426 | 1.0000 | **+0.0574** |
| Recall@10 | 0.9580 | 1.0000 | **+0.0420** |
| MRR@10 | 0.8472 | 0.9791 | **+0.1319** |
| nDCG@10 | 0.8738 | 0.9845 | **+0.1107** |

For teaching purposes, note the difference between **absolute** and **relative**
improvement.

For example, MRR@10 increased by `0.1319`, i.e. **13.19 percentage points** in
absolute terms. This is not the same statement as saying that it increased by
13.19 percent relative to its baseline value.

These are strong validation gains, but they are still checkpoint-selection results.
The test set is required before making the final generalisation claim.

In [25]:
comparison_rows = []

for row in history:
    comparison = {
        "epoch": row["epoch"]
    }

    for metric in [
        "recall_at_1",
        "recall_at_5",
        "recall_at_10",
        "mrr_at_10",
        "ndcg_at_10",
    ]:
        baseline_value = baseline_validation_metrics[metric]
        current_value = row[metric]

        comparison[f"baseline_{metric}"] = baseline_value
        comparison[f"lora_{metric}"] = current_value
        comparison[f"delta_{metric}"] = (
            current_value - baseline_value
        )

    comparison_rows.append(comparison)

comparison_df = pd.DataFrame(comparison_rows)

display(comparison_df)

,epoch,baseline_recall_at_1,lora_recall_at_1,delta_recall_at_1,baseline_recall_at_5,lora_recall_at_5,delta_recall_at_5,baseline_recall_at_10,lora_recall_at_10,delta_recall_at_10,baseline_mrr_at_10,lora_mrr_at_10,delta_mrr_at_10,baseline_ndcg_at_10,lora_ndcg_at_10,delta_ndcg_at_10
0,1,0.706331,0.878165,0.171835,0.942636,1.0,0.057364,0.95801,1.0,0.04199,0.847211,0.973299,0.126088,0.87379,0.980189,0.106399
1,2,0.706331,0.888501,0.182171,0.942636,1.0,0.057364,0.95801,1.0,0.04199,0.847211,0.979113,0.131902,0.87379,0.984501,0.110711


## 25. Select the best adapter

The output reports:

```text
Best epoch: 2
Best validation MRR@10: 0.9791
Baseline validation MRR@10: 0.8472
```

The selected adapter is therefore the one saved under:

```text
models/bge-m3-legal-pt-lora/best
```

The selection criterion is explicit and reproducible:

```text
highest semantic validation MRR@10
```

Epoch 2 is not selected merely because it is the last epoch or because its training
loss is lower.

In [26]:
print("Best epoch:", best_epoch)

print(
    f"Best validation {MODEL_SELECTION_METRIC}: "
    f"{best_metric:.4f}"
)

print(
    f"Baseline validation {MODEL_SELECTION_METRIC}: "
    f"{baseline_validation_metrics[MODEL_SELECTION_METRIC]:.4f}"
)

if best_epoch == 0:
    print(
        "\nNo LoRA checkpoint improved "
        "the original baseline."
    )
else:
    print("\nBest adapter:")
    print(BEST_MODEL_DIR.resolve())

Best epoch: 2
Best validation mrr_at_10: 0.9791
Baseline validation mrr_at_10: 0.8472

Best adapter:
C:\Users\user\Documents\GitHub\legal-embeddings-finetuning-pt\models\bge-m3-legal-pt-lora\best


## 26. Save result tables

The notebook saves:

```text
training_history.csv
validation_comparison.csv
```

The first preserves the metrics and timings of each epoch.

The second preserves the direct comparison between the original BGE-M3 baseline
and every LoRA checkpoint.

Keeping these tables avoids relying on notebook output alone when documenting or
analysing the experiment later.

In [27]:
HISTORY_PATH = (
    RESULTS_DIR / "training_history.csv"
)

COMPARISON_PATH = (
    RESULTS_DIR / "validation_comparison.csv"
)

history_df.to_csv(
    HISTORY_PATH,
    index=False
)

comparison_df.to_csv(
    COMPARISON_PATH,
    index=False
)

print(HISTORY_PATH.resolve())
print(COMPARISON_PATH.resolve())

C:\Users\user\Documents\GitHub\legal-embeddings-finetuning-pt\results\finetuning_lora\training_history.csv
C:\Users\user\Documents\GitHub\legal-embeddings-finetuning-pt\results\finetuning_lora\validation_comparison.csv


## 27. Save experiment metadata

`finetuning_metadata.json` records the configuration needed to interpret this run.

This includes model, device, random seed, sequence length, optimiser-related
settings, LoRA parameters, trainable parameter counts, baseline metrics and the
selected epoch.

The combination of:

```text
dataset
+ source code
+ model/adapters
+ configuration
+ metrics
```

is what makes an experiment reproducible.

In [28]:
metadata = {
    "experiment_version": "3.0-lora",
    "training_method": "LoRA/PEFT",
    "base_model": MODEL_NAME,
    "device": DEVICE,
    "gpu": torch.cuda.get_device_name(0),
    "random_seed": RANDOM_SEED,
    "max_seq_length": MAX_SEQ_LENGTH,
    "train_batch_size": TRAIN_BATCH_SIZE,
    "encode_batch_size": ENCODE_BATCH_SIZE,
    "epochs": NUM_EPOCHS,
    "learning_rate": LEARNING_RATE,
    "warmup_ratio": WARMUP_RATIO,
    "weight_decay": WEIGHT_DECAY,
    "triplet_margin": TRIPLET_MARGIN,
    "negatives_per_query": NEGATIVES_PER_QUERY,
    "semantic_queries_only": True,
    "mixed_precision": USE_AMP,
    "lora_r": LORA_R,
    "lora_alpha": LORA_ALPHA,
    "lora_dropout": LORA_DROPOUT,
    "lora_target_modules": LORA_TARGET_MODULES,
    "trainable_parameters": trainable_params,
    "total_parameters": total_params,
    "trainable_percentage": trainable_ratio,
    "training_queries": len(semantic_records),
    "training_triplets": len(training_records),
    "validation_semantic_queries": len(
        semantic_validation_queries
    ),
    "model_selection_metric": MODEL_SELECTION_METRIC,
    "baseline_validation_metrics": (
        baseline_validation_metrics
    ),
    "best_epoch": best_epoch,
    "best_validation_metric": best_metric,
    "best_adapter_path": (
        str(BEST_MODEL_DIR)
        if best_epoch > 0
        else None
    ),
}

METADATA_PATH = (
    RESULTS_DIR / "finetuning_metadata.json"
)

with METADATA_PATH.open(
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        metadata,
        f,
        ensure_ascii=False,
        indent=2
    )

print(METADATA_PATH.resolve())

C:\Users\user\Documents\GitHub\legal-embeddings-finetuning-pt\results\finetuning_lora\finetuning_metadata.json


## 28. Reloading the trained LoRA model

The saved adapter is not a complete standalone copy of BGE-M3.

To reconstruct the adapted embedding model later:

```text
BAAI/bge-m3
      +
best LoRA adapter
      =
fine-tuned legal embedding model
```

This is why LoRA checkpoints are much smaller than saving a new full model for
every epoch.

The final evaluation notebook must load the original base model and then attach
the `best` adapter before encoding the test corpus and queries.

## 29. Final CUDA memory report

At the end of the completed run:

```text
Free VRAM   2.74 GB
Allocated   2.14 GB
Reserved    2.18 GB
Total       6.00 GB
```

The values are close to those measured before training, which is consistent with
the model and adapter still being resident in memory.

`torch.cuda.empty_cache()` can release unused cached blocks, but it cannot release
live tensors that are still referenced by the notebook.

In [29]:
gc.collect()
torch.cuda.empty_cache()

free, total = torch.cuda.mem_get_info()

print(f"Free VRAM : {free / 1024**3:.2f} GB")
print(f"Total VRAM: {total / 1024**3:.2f} GB")
print(
    f"Allocated : "
    f"{torch.cuda.memory_allocated() / 1024**3:.2f} GB"
)
print(
    f"Reserved  : "
    f"{torch.cuda.memory_reserved() / 1024**3:.2f} GB"
)

Free VRAM : 2.74 GB
Total VRAM: 6.00 GB
Allocated : 2.14 GB
Reserved  : 2.18 GB


## 30. Conclusions from this training run

The completed experiment provides three distinct findings.

### 1. LoRA made local BGE-M3 fine-tuning feasible

Only **1,179,648 of 568,934,400 parameters** were trainable — approximately
**0.2073%** of the model.

The experiment completed on an RTX 4050 Laptop GPU with 6 GB of VRAM.

### 2. Semantic validation retrieval improved substantially

The strongest checkpoint was epoch 2:

```text
                         Baseline      LoRA epoch 2
Recall@1                  0.7063          0.8885
Recall@5                  0.9426          1.0000
Recall@10                 0.9580          1.0000
MRR@10                    0.8472          0.9791
nDCG@10                   0.8738          0.9845
```

In particular:

```text
MRR@10: 0.8472 → 0.9791
absolute gain: +0.1319
```

### 3. Epoch 2 is the selected checkpoint

Epoch 1 already produced a large improvement, but epoch 2 increased semantic
validation `MRR@10` from **0.9733 to 0.9791** and was therefore saved as `best`.

## What we still cannot conclude

We should **not yet state that the final model generalises better to unseen test
data**.

The validation set influenced model selection. It told us which epoch to keep.

The final experimental sequence must therefore be:

```text
TRAIN
  ↓
learn LoRA parameters

VALIDATION
  ↓
select epoch 2

TEST
  ↓
evaluate once
  ↓
final conclusion
```

## Next notebook — 07 Final Evaluation

The next stage should load both:

```text
Original BGE-M3
BGE-M3 + best LoRA adapter
```

and evaluate them on the exact same untouched test set.

The final report should compare:

- Recall@1, Recall@5 and Recall@10;
- MRR@10;
- nDCG@10;
- semantic results;
- results by legal document;
- absolute improvements;
- examples where LoRA improves or worsens the ranking.

Only that held-out comparison can establish whether the domain adaptation
generalises beyond the data used for training and checkpoint selection.